In this notebook, we start with the Lindblad master equation

$$ \partial_t \rho = -i[\varepsilon_d(b+b^\dagger)/2+g(a^2b^\dagger+h.c.), \rho]+\kappa_b[b\rho b^\dagger - 1/2\{b^\dagger b,\rho\}]$$

This equation leads to Keldysh action and these equations of motion
\begin{cases}
\dot A = -\sqrt 2 g(AB-XY)\\
\dot X = -\sqrt 2 g(AY-XB)\\
\dot B = -\varepsilon_d /2 + g(A^2+X^2)/\sqrt 2-\kappa_b (B/2+Y)\\
\dot Y = -\sqrt 2 g AX + \kappa_b Y/2\\
\end{cases}

that can be found through this Hamiltonian

$$H=-\sqrt 2 g ABX+g(A^2+X^2)Y/\sqrt 2-\varepsilon_d Y/\sqrt 2 - \kappa_b (B+Y)Y/2.$$

There are four fixed points
\begin{cases}
A = 0; \qquad X = 0; \qquad B=B_0 = -\sqrt{2}\varepsilon_d/\kappa_b; \qquad Y =0\\
A^2 = A_0^2 = \varepsilon_d/g; \qquad X = 0; \qquad B=0;  \qquad Y =0\\
A = 0; \qquad X^2 = A_0^2; \qquad B=0; \qquad Y =0\\
A^2 = A_0^2/8; \qquad X = A; \qquad B = Y = B_0 /4\\
\end{cases}

We are looking for the optimal path from $(A=A_0, X=0, B=0, Y=0)$ to $(A=0, X=0, B=B_0, Y=0)$, so that $H=0$ on this path. In this subspace, we get

$$B=\frac{g(A^2+X^2)Y-\varepsilon_d Y-\kappa_bY^2/\sqrt{2}}{2gAX+\kappa_bY/\sqrt 2} ;$$

which is singular when $\dot Y=0$ !

In [ ]:
include("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/config/requirements.jl");

In [ ]:
const ∞ = Inf
ħ=1
e=1
β = ∞ # Case zero temperature T=0, β → ∞
ϕ_quantum=1/2
#### Caracterization of the circuit #####
ω_a = 25.338776456203686
ω_b = 2*ω_a

φ_a = 0.11
Z_a = 2*ϕ_quantum^2*φ_a^2
C_a_inv = Z_a*ω_a
E_Ca = C_a_inv/2

φ_b = 0.204
Z_b = 2*ϕ_quantum^2*φ_a^2
C_b_inv = Z_b*ω_b
E_Cb = C_b_inv/2

E_J = 37.12*2pi/2

#Environment
κ_b=5/10.4; # 1/ns from Leon's paper

ω_d=ω_b # Drive frequency
ω_p=2*ω_a-ω_d # Pump Frequency

ε_p = 0.1
g2 = besselj(1,ε_p)*E_J*φ_a^2*φ_b # (Leon's paper)
ε_d = 10.12*g2#0.75*g2_max

g = E_J*sin(ε_p)*φ_a^2*φ_b
params = (g,ε_d,κ_b);

### Defining the fixed points

In [ ]:

A_0 = sqrt(ε_d/g)
B_0 = -sqrt(2)*ε_d/κ_b
fixed_points_list = [(0,0,B_0,0),(A_0,0,0,0),(-A_0,0,0,0),(0,A_0,0,0),(0,-A_0,0,0),(A_0/sqrt(8),A_0/sqrt(8),B_0/4,B_0/4),(-A_0/sqrt(8),-A_0/sqrt(8),B_0/4,B_0/4)]

function Hamiltonian(var,params)
    A,X,B,Y = var
    g,ε_d,κ_b = params
    return -sqrt(2)*g*A*B*X+g/sqrt(2)*(A^2+X^2)*Y-ε_d/sqrt(2)*Y-κ_b/2*(B+Y)*Y
end

function B_from_vars(vars, params)
    A,X,B,Y = var
    g,ε_d,κ_b = params
    return (g*(A^2+X^2)*Y-ε_d*Y-κ_b/sqrt(2)*Y^2)/(2*g*A*X+κ_b/sqrt(2)*Y)
end

In [ ]:
for fp in fixed_points_list
    println(Hamiltonian(fp,params))
end

### We only keep the first five fixed points

In [ ]:
A = range(-2A_0, 2A_0, length=20)
X = range(-2A_0, 2A_0, length=20)
Y = range(2B_0, -2B_0, length=20)


points = [(a, x, y) for a in A, x in X, y in Y]

xs = vec([p[1] for p in points])
ys = vec([p[2] for p in points])
zs = vec([p[3] for p in points])
scatter3d(xs, ys, zs, markersize=.1)

scatter3d!(
    [fp[1] for fp in fixed_points_list[1:5]],
    [fp[2] for fp in fixed_points_list[1:5]],
    [fp[4] for fp in fixed_points_list[1:5]],
    markersize=5,
    color=:red
)
xlabel!("A")
ylabel!("X")
zlabel!("Y")


In [ ]:
function equation!(du, u, params, t)
    g, ε_d, κ_b = params

    A = u[1]
    X = u[2]
    Y = u[3]

    # Default: compute B from the H=0 constraint when possible
    denom = 2g*A*X + κ_b*Y/sqrt(2)

    if abs(denom) < 1e-10
        # special zero-energy fixed point
        if abs(A) < 1e-10 && abs(X) < 1e-10 && abs(Y) < 1e-10
            B = -sqrt(2) * ε_d / κ_b
        else
            error("Singular denominator in B(A,X,Y): cannot eliminate B on this branch.")
        end
    else
        B = (g*(A^2 + X^2)*Y - ε_d*Y - κ_b*Y^2/sqrt(2)) / denom
    end

    du[1] = -sqrt(2) * g * (A*B - X*Y)
    du[2] = -sqrt(2) * g * (A*Y - X*B)
    du[3] =  sqrt(2) * g * A*X + κ_b*Y/2

    return nothing
end

In [ ]:
t_min = 0.0
t_max = 100/κ_b
tspan = (t_min, t_max)
A_init, X_init, Y_init = 1, 2, 0
u_0 = [A_init, X_init, Y_init]
prob = ODEProblem(equation!, u_0, tspan, params)
sol = solve(prob, Tsit5())
t = sol.t
A = [u[1] for u in sol.u]
X = [u[2] for u in sol.u]
Y = [u[3] for u in sol.u];


In [ ]:
#scatter3d(A,X,Y, ms=2)
scatter3d(
    [fp[1] for fp in fixed_points_list[1:5]],
    [fp[2] for fp in fixed_points_list[1:5]],
    [fp[4] for fp in fixed_points_list[1:5]],
    markersize=5,
    color=:red
)
xlabel!("A")
ylabel!("P_A")
zlabel!("P_B")

In [ ]:

# Second-order ODE:
#   A¨ - (κ_b/2) A˙ + 2g^2 A0 A (A0 - A) = 0
#
# Rewritten as a first-order system with
#   u[1] = A
#   u[2] = V = A˙
function eom_second_order!(du, u, p, t)
    g, A0, κ_b = p
    A, V = u

    du[1] = V
    du[2] = (κ_b / 2) * V + 2 * g^2 * A0 * A * (A0 - A)
end

# Initial condition:
# start slightly away from A=A0 with a small negative velocity
# so that A decreases toward 0
δA = 1e-2
V0 = -1e-3
u0 = [A0 - δA, V0]

tspan = (0.0, 34.0)
params = (g, A0, κ_b)

prob = ODEProblem(eom_second_order!, u0, tspan, params)
sol = solve(prob, Tsit5(); reltol=1e-9, abstol=1e-9)

A = [u[1] for u in sol.u]

# Print where the trajectory ends
println("Final point:")
println("A(t_f) = ", sol[end][1])
println("Target fixed point is approximately (0)")


In [ ]:
plot(sol.t,A)
hline!([A_0])
ylabel!("A")
xlabel!("t")

In [ ]:
function eom_1!(du, u, p, t)
    g, ε_d, κ_b = p
    A, Y = u

    du[1] = sqrt(2) * g * A_0 * Y
    du[2] = sqrt(2) * g * A * (A_0-A) + κ_b*Y/2
end

# Start slightly away from the unstable fixed point (A0, 0)
u0 = [A_0 - 1e-3, -1e-3]

# Time span
tspan = (0.0, 40.0)

# Solve
prob = ODEProblem(eom_1!, u0, tspan, (g, ε_d, κ_b))
sol = solve(prob, Tsit5(); reltol=1e-9, abstol=1e-9)

A = [u[1] for u in sol.u]
Y = [u[2] for u in sol.u]

# Print where the trajectory ends
println("Final point:")
println("A(t_f) = ", sol[end][1])
println("Y(t_f) = ", sol[end][2])
println("Target fixed point is approximately (0, 0)")


In [ ]:
plot(A,Y)
vline!([A_0])
hline!([0])
xlabel!("A")
ylabel!("Y")

In [ ]:
# ODE system:
#   dX/dt = sqrt(2) g X B
#   dB/dt = -ε_d/sqrt(2) + g X^2/sqrt(2) - κ_b B/2
function eom_2!(du, u, p, t)
    g, ε_d, κ_b = p
    X, B = u

    du[1] = sqrt(2) * g * X * B
    du[2] = -ε_d / sqrt(2) + g * X^2 / sqrt(2) - κ_b * B / 2
end

# Start slightly away from the unstable fixed point (A0, 0)
u0 = [A_0 - 1e-2, -1e-6]

# Time span
tspan = (0.0, 500.0)

# Solve
prob = ODEProblem(eom_2!, u0, tspan, (g, ε_d, κ_b))
sol = solve(prob, Tsit5(); reltol=1e-9, abstol=1e-9)

X = [u[1] for u in sol.u]
B = [u[2] for u in sol.u]

# Print where the trajectory ends
println("Final point:")
println("X(t_f) = ", sol[end][1])
println("B(t_f) = ", sol[end][2])
println("Target fixed point is approximately (0, $B0)")


In [ ]:
plot(X,B)
vline!([A_0])
hline!([B_0])
xlabel!("X")
ylabel!("B")

After rescaling, we focus on these equations of motion
\begin{equation}
    \begin{cases}
        \dot  A = P_AP_B-AB\\
        \dot P_A = P_AB-AP_B\\
        \dot B  = \frac{\kappa_b^2}{4\varepsilon_dg}(A^2+P_A^2-B-2P_B-1)\\
        \dot P_B = \frac{\kappa_b^2}{4\varepsilon_dg}(2AP_A+P_B),
    \end{cases}
\end{equation}
and the fixed points for $H=0$ are
\begin{equation}
    \begin{cases}
        P_A=P_B=0 \qquad B=0 \text{ and }A=\pm1\\
        P_A=P_B=0 \qquad A=0 \text{ and }B=-1\\
        B=P_B = 0 \qquad A=0 \text{ and }P_A=\pm1.
    \end{cases}
\end{equation}
Thus a try for the instanton path is $(A=1, B=0, P_A=0, P_B=0)\to(A=0, B=0, P_A=1, P_B=0)\to(A= 0, B=-1, P_A=0, P_B=0)$.

Let's focus on the second part, we fix $A=0$ and $P_B=0$, so that
\begin{equation}
    \begin{cases}
        \dot P_A = P_AB\\
        \dot B  = \frac{\kappa_b^2}{4\varepsilon_dg}(P_A^2-B-1).
    \end{cases}
\end{equation}

In [ ]:
# ODE system:
#   dX/dt = sqrt(2) g X B
#   dB/dt = -ε_d/sqrt(2) + g X^2/sqrt(2) - κ_b B/2
function eom_2_rescaled!(du, u, p, t)
    P_A, B = u

    du[1] =  P_A * B
    du[2] = p*(P_A^2-B-1)
end

# Start slightly away from the unstable fixed point (A0, 0)
u0 = [1 - 1e-2, -1e-6]

# Time span
tspan = (0.0, 300.0)
p = κ_b^2/(4*ε_d*g)
# Solve
prob = ODEProblem(eom_2_rescaled!, u0, tspan, p)
sol = solve(prob, Tsit5(); reltol=1e-9, abstol=1e-9)

P_A = [u[1] for u in sol.u]
B = [u[2] for u in sol.u]

# Print where the trajectory ends
println("Final point:")
println("P_A(t_f) = ", sol[end][1])
println("B(t_f) = ", sol[end][2])
println("Target fixed point is approximately (0, -1)")


In [ ]:
plot(P_A,B)
vline!([1])
hline!([-1])
xlabel!("X")
ylabel!("B")

In [ ]:
# ODE system:
#   dX/dt = sqrt(2) g X B
#   dB/dt = -ε_d/sqrt(2) + g X^2/sqrt(2) - κ_b B/2
function eom_1_rescaled!(du, u, p, t)
    A, P_B = u 

    du[1] = P_B
    du[2] = p*(2*A*(1-A)+P_B)
end

# Start slightly away from the unstable fixed point (A0, 0)
u0 = [1.0-1e-2, -23.84236939836711677/1000]

# Time span
tspan = (0.0, 2.8)
p = κ_b^2/(4*ε_d*g)
# Solve
prob = ODEProblem(eom_1_rescaled!, u0, tspan, p)
sol = solve(prob, Tsit5(); reltol=1e-9, abstol=1e-9)

A = [u[1] for u in sol.u]
P_B = [u[2] for u in sol.u]

# Print where the trajectory ends
println("Final point:")
println("A(t_f) = ", sol[end][1])
println("P_B(t_f) = ", sol[end][2])
println("Target fixed point is approximately (0, 0)")

# define vector field
function flow(A, P_B)
    dA = P_B
    dP_B = p*(2*A*(1-A)+P_B)
    return dA, dP_B
end

# grid
A_range = LinRange(-.1, 1.1, 30)
P_B_range = LinRange(-.7, .1, 30)

A_grid = [a for a in A_range, pb in P_B_range]
P_B_grid = [pb for a in A_range, pb in P_B_range]

dA = similar(A_grid)
dP_B = similar(P_B_grid)

for i in eachindex(A_grid)
    dA[i], dP_B[i] = flow(A_grid[i], P_B_grid[i])
end

# normalize arrows for better visualization
norm = sqrt.(dA.^2 .+ dP_B.^2 .+ 1e-8)
dA ./= norm*10
dP_B ./= norm*10;

In [ ]:
# plot
quiver(
    A_grid, P_B_grid,
    quiver=(dA, dP_B),
    quiver_scale=0.2,
    xlabel="A", ylabel="P_B",
    title="Flow field",
    label="",
    color =:blue,
    lw=.5,
)
scatter!(A, P_B, color=:red, label="Try path")

A_plot = 0:.01:1
plot!(A_plot, 2 .* (A_plot .-1/2).^2 .-1/2, label="Adiabatic solution")

# mark fixed points
scatter!([1, 0], [0, 0], markersize=6, label="Fixed points")
ylims!(-.7,0.4)
#savefig("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/figures/switching_rate/path_adiabatic_ansatz_almost_adiabatic_20_03_2026.pdf")

In [ ]:
# define vector field
function flow(VA, VB)
    dVA = -VB
    dVB = p * (VA^2 + VB - 1)
    return dVA, dVB
end

# grid
VA_range = LinRange(-1.1, 1.1, 35)
VB_range = LinRange(-0.2, 1.3, 35)

VA_grid = [va for va in VA_range, vb in VB_range]
VB_grid = [vb for va in VA_range, vb in VB_range]

dVA = similar(VA_grid)
dVB = similar(VB_grid)

for i in eachindex(VA_grid)
    dVA[i], dVB[i] = flow(VA_grid[i], VB_grid[i])
end

# normalize arrows for better visualization
norm = sqrt.(dVA.^2 .+ dVB.^2 .+ 1e-8)
dVA ./= norm*10
dVB ./= norm*10

# plot
quiver(
    VA_grid, VB_grid,
    quiver=(dVA, dVB),
    quiver_scale=0.2,
    xlabel="V_A", ylabel="V_B",
    title="Flow field",
    aspect_ratio=1,
    legend=false,
    color =:blue,
    lw=.5,
)

# mark fixed points
scatter!([1, -1], [0, 0], markersize=6)

# We set-up a simple time-independent two-well system

We start with the usual equation of the resonant flux-biased ATS Hamiltonian in the RWA
\begin{equation}
    \begin{cases}
    \dot a = -i\beta_a'a^*b\\
    \dot b = -i\beta_b'a^2/2-i\varepsilon_d'/2-\kappa_b'b
    \end{cases}
\end{equation}
Then, we perform adiabatic elimination $\dot b = 0$ and we get $$\dot a = -i\beta_a'a^*(i\beta_b'a^2/2\kappa_b'-i\varepsilon_d'/2\kappa_b')$$

We introduce the quadratures $x=(a+a^*)/\sqrt 2$ and $p=i(a^*-a)/\sqrt 2$ 
\begin{equation}
    \begin{cases}
    \dot x = - \aleph (x^2+p^2)x - \beth x\\
    \dot p = - \aleph (x^2+p^2)p + \beth p
    \end{cases},
\end{equation}
with $\aleph = \beta_a'\beta_b'/4\kappa_b'$ and $\beth = \beta_a'\varepsilon_d'/2\kappa_b'$.

Then we can define a potential $U(x,p)$ so that $\dot x = \partial_xU$ and $\dot p = \partial_pU$ :$$U(x,p) = \aleph(x^2+p^2)^2/4+\beth(x^2-p^2)/2$$



In [ ]:
const ∞ = Inf
ħ=1
e=1
β = ∞ # Case zero temperature T=0, β → ∞
ϕ_quantum=1/2
#### Caracterization of the circuit #####
ω_a = 25.338776456203686
ω_b = 2*ω_a

φ_a = 0.11
Z_a = 2*ϕ_quantum^2*φ_a^2
C_a_inv = Z_a*ω_a
E_Ca = C_a_inv/2

φ_b = 0.204
Z_b = 2*ϕ_quantum^2*φ_a^2
C_b_inv = Z_b*ω_b
E_Cb = C_b_inv/2

E_J = 37.12*2pi*1.5

#Environment
κ_b=1/10.4; # 1/ns from Leon's paper

ω_d=ω_b # Drive frequency
ω_p=2*ω_a-ω_d # Pump Frequency

ε_p = 0.2
g2 = besselj(1,ε_p)*E_J*φ_a^2*φ_b # (Leon's paper)
ε_d = 10.12*g2#0.75*g2_max

β_a_p, β_b_p = 2*E_J*sin(ε_p)*φ_a^2/(ω_a), 2*E_J*sin(ε_p)*φ_b^2/(ω_b)
ε_d_p = ε_d*φ_b/ω_b
κ_b_p = κ_b / ω_b

ℵ = β_a_p*β_b_p/(4*κ_b_p)
ℶ = β_a_p*ε_d_p/(2*κ_b_p)

In [ ]:
sqrt(2*ε_d_p/β_b_p)

In [ ]:
function equation!(du,u,params,t)
    ℵ,ℶ = params

    x = u[1]
    p = u[2]
        
    du[1] = -ℵ*(x^2+p^2)*x-ℶ*x
    du[2] = -ℵ*(x^2+p^2)*p+ℶ*p
    return nothing
end

In [ ]:
t_min = 0.0
t_max = 2/κ_b_p 
tspan = (t_min, t_max)
x_0, p_0 = 0.0, -0.001
u_0 = [x_0, p_0]
prob = ODEProblem(equation!, u_0, tspan, (ℵ,ℶ))
sol = solve(prob, Tsit5())
t = sol.t
x = [u[1] for u in sol.u]
p = [u[2] for u in sol.u]
p_stable=p;

In [ ]:
x_all = -.2:0.01:.2
p_all = -.35:0.01:.35

U = [ℵ*(x^2 + p^2)^2/4 + ℶ*(x^2 - p^2)/2 for p in p_all, x in x_all]

heatmap(x_all, p_all, U,
    colorbar_title="\n Meta-potential",
    xlabel=L"$x$",
    ylabel=L"$p$",
)
scatter!(x,p, label="Trajectory")

Since the system is only 1D ($x=0$ on the minimizing path and on the dissipative path), we will focus only on $p$. Therfore the equation of motion reads
$$\dot p = -\aleph p^3+\beth p$$

# Writing the Hamiltonian and initializing the GMAM

We now add noise to the system
\begin{equation}
    \dot p = - \aleph p^3 + \beth p + \sqrt \epsilon\eta(t).
\end{equation}

We can write the associated action: $$S=\int dt [P_p\dot p - H(p,P_p)],$$
with the Hamiltonian $$H(p,P_p) = [ - \aleph p^2+\beth]pP_p + \epsilon P_p^2$$

We parametrize time to solve a geometric problem $$\min_{\lambda\geq0}\min_p\max_P\int_0^1[p'P-\lambda^{-1}H(p,P)]ds,$$
and we add a constraint on the parametrization of $p$ : $|dp/ds|=|p'|=cst$

Inputs: $M = 100 \in \mathbb N$, two stable fixed points $\phi_a = (-\sqrt{2\varepsilon_d'/\beta_b'})$ and $\phi_b = (0)$ of the noiseless dynamics where $H(\phi_{a,b},0)=\nabla_{\bm \theta}H(\phi_{a,b},0)=0$; a path $\{\hat\phi_i^0\}_{i\in I}$ with $\hat\phi_0^0 =\phi_a$ and $\hat\phi_M^0 =\phi_b$, such that $|\hat\phi_{i+1}^0-\hat\phi_{i}^0|$ is constant in $i$; the functions $f(\mathbf u, \mathbf v)= \alpha \nabla_{\bm \phi}H - \nabla_{\bm \theta}H$, $g(\mathbf u, \mathbf v)=\alpha \nabla_{\bm \phi}H + \nabla_{\bm \theta}H$ and $\lambda(\mathbf u, \mathbf v) = \biggl[|\langle \nabla_{\bm \theta}H, \partial_s \hat\phi \rangle|+\sqrt{\max(0,\langle \nabla_{\bm \theta}H, \partial_s \hat\phi \rangle^2-4H|\partial_s\hat\phi|^2)}\biggr]/(2|\partial_s\hat\phi|^2)$; $\Delta\tau = 0.1 > 0, \alpha = 1 >0$

We have $\nabla_{\bm\phi}H = \begin{pmatrix}- 3\aleph p^2P_p +\beth P_p \end{pmatrix}$ and $\nabla_{\bm\theta}H = \begin{pmatrix} -\aleph p^3 +\beth p + 2 \epsilon P_p\end{pmatrix}$

# Let's perform the gradient descent ascent

We got that for $n\geq0$, we solve for each $i=M,M-1,\dots,1,0$ for the value of $u_i^{n+1}$ through $$u_M^{n+1} = -v_M^n+2\phi_b,$$ $$ u_i^{n+1} = \biggl[\lambda_i(u^n,v^n)u_{i+1}^{n+1}/\Delta s + f(u_i^n,v_i^n) + u_i^n/\Delta \tau\biggr]/\biggl[1/\Delta\tau + \lambda_i(u^n,v^n)/\Delta s\biggr] , \qquad i=M-1,\dots,1,0$$

For the value of $v_i^{n+1}$ through $$v_0^{n+1} = -u_0^{n+1}+2\phi_a,$$ $$ v_{i}^{n+1} = \biggl[\lambda_i(u^{n+1},v^n)v_{i-1}^{n+1}/\Delta s + g(u_i^{n+1},v_i^n)+v_i^n/\Delta\tau\biggr]/\biggl[1/\Delta\tau + \lambda_i(u^{n+1},v^n)/\Delta s\biggr] , \qquad i=1,\dots,M-1,M$$

In [ ]:
# ODE system:
#   dX/dt = sqrt(2) g X B
#   dB/dt = -ε_d/sqrt(2) + g X^2/sqrt(2) - κ_b B/2
function eom_test!(du, u, p, t)
    A, P_A = u

    du[1] =  A- A^3-3*A*P_A^2-4*A^2*P_A
    du[2] =  -P_A+ P_A^3+3*P_A*A^2+4*P_A^2*A
end

# Start slightly away from the unstable fixed point (A0, 0)
u0 = [1-1e-10, 1e-10]

# Time span
tspan = (0.0, 20.0)
p = κ_b^2/(4*ε_d*g)
# Solve
prob = ODEProblem(eom_test!, u0, tspan, p)
sol = solve(prob, Tsit5(); reltol=1e-9, abstol=1e-9)

A = [u[1] for u in sol.u]
P_A = [u[2] for u in sol.u]

# Print where the trajectory ends
println("Final point:")
println("A(t_f) = ", sol[end][1])
println("P_A(t_f) = ", sol[end][2])
println("Target fixed point is approximately (0, 1)")


In [ ]:
plot(A, P_A, label="", linewidth=2, aspect_ratio =:equal)
vline!([1], linestyle=:dash, color=:black, label="")
hline!([1], linestyle=:dash, color=:black, label="")
xlabel!("A")
ylabel!("P_A")

In [ ]:
A_all = -1.2:0.01:1.2
P_A_all = -1.2:0.01:1.2

H_vals = [A*P_A-A^3*P_A-A*P_A^3-2*A^2*P_A^2 for P_A in P_A_all, A in A_all]

heatmap(A_all, P_A_all, H_vals,
    colorbar_title="\n Hamiltonian",
    xlabel=L"$A$",
    ylabel=L"$P_A$",
)
contour!(A_all, P_A_all, H_vals; levels=[0.0], c=:white, lw=2)
scatter!(A, P_A, label="", ms=1)